<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/autogen_core_support_desk_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Agents with AutoGen — Core Concepts, Hands-On

**Use case: a mini customer-support desk.** A customer sends a ticket. One agent decides
*what kind of problem it is*, a second agent *looks up real data with tools and drafts a reply*,
and a third agent *collects the finished replies*.

That tiny application is enough to show all five ideas from the class:

| Class topic | Where you see it in this notebook |
|---|---|
| **Goals** | The `SystemMessage` each agent carries — its job description |
| **Actions** | `FunctionTool` — Python functions the model is allowed to call |
| **AutoGen Agents** | `RoutedAgent` + `@message_handler` — objects that react to messages |
| **Environments** | `SingleThreadedAgentRuntime`, topics and subscriptions — the world agents live in |
| **Applications** | The three agents wired together, end to end |

### How to read this notebook
Every code cell is preceded by a box that tells you three things:

- **What this cell does** — the idea in one or two lines
- **Input** — what goes in (a value, a message, a previous cell)
- **Output** — what you should see when it runs

### Before you start
- Python 3.10 or newer
- A **Groq API key** — create one free at [console.groq.com/keys](https://console.groq.com/keys).
  Groq has a free tier, so no credits or card are needed for this notebook.
- The model we use is `openai/gpt-oss-120b` — an **open-weights** model hosted by Groq
  (the name says "openai" because that is who released the weights; it is not the OpenAI API)
- Roughly 15-20 model calls in total

---
## Part 0 — Setup

### Cell 1 — Install the libraries

**What this cell does:** installs the two packages we need. `autogen-core` is the agent
framework itself (agents, runtime, tools). `autogen-ext[openai]` adds the OpenAI-style model
client — which is also how we talk to **Groq**, because Groq exposes an OpenAI-compatible API.
There is no separate "Groq package" to install.

**Input:** nothing — it just talks to PyPI.

**Output:** pip progress lines ending in `Successfully installed autogen-core-0.7.x ...`.
If they are already installed you will see `Requirement already satisfied`.

> Restart the kernel after this cell **only** if the import in Cell 3 fails.

In [ ]:
# Installs the two libraries this notebook needs. ("-q" just means "print less".)
#   autogen-core        -> the agent framework itself: agents, messages, tools, the runtime
#   autogen-ext[openai] -> the connector for OpenAI-style APIs. Groq speaks that same
#                          language, so this one package covers Groq too.
%pip install -q "autogen-core>=0.7,<0.8" "autogen-ext[openai]>=0.7,<0.8"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 9.0 MB/s eta 0:00:00


### Cell 2 — Provide your Groq API key

**What this cell does:** stores your key in the environment variable `GROQ_API_KEY`.
`getpass` hides the key while you type, so it is never saved into the notebook file — important
if you share the `.ipynb` afterwards.

Get a key at [console.groq.com/keys](https://console.groq.com/keys). Keys start with `gsk_`.

**Input:** your Groq API key, typed into the prompt box.

**Output:**
```
Key loaded: True
```

In [ ]:
# ===== SETUP 1 of 2: hand the notebook your Groq API key =====

import os        # "os" = operating system. Lets Python read/write environment variables,
                 # which are just named settings that live in this running session.
import getpass   # "getpass" asks you to type a secret and HIDES the characters as you type,
                 # so your key is never visible on screen and never saved into this file.

# os.environ is a dictionary of those settings. .get() returns None if the name is missing.
# So this reads as: "if the key is NOT already set, ask me for it."
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key (gsk_...): ")

# bool(...) turns the key into just True or False, so we can confirm it arrived
# WITHOUT ever printing the secret itself.
print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

Enter your Groq API key (gsk_...): ··········
Key loaded: True


### Cell 3 — Imports

**What this cell does:** pulls in everything we will use, grouped by purpose:

- `autogen_core` — the runtime, agent base class, ids, topics
- `autogen_core.models` — the message types an LLM understands
- `autogen_core.tools` — turning Python functions into callable tools
- `autogen_ext.models.openai` — the OpenAI-compatible model client (we point it at Groq)

**Input:** the installed packages from Cell 1.

**Output:**
```
autogen-core version: 0.7.5
```

In [ ]:
# ===== SETUP 2 of 2: bring in the tools we will use =====
# An "import" is like taking a tool out of a toolbox before starting work.
# Nothing runs yet - this cell only makes names available to the cells below.

import asyncio                      # lets slow work (network calls) wait without freezing everything
import json                         # converts between Python dictionaries and JSON text
import os                           # environment variables again - we read GROQ_API_KEY below
from dataclasses import dataclass   # shortcut for simple "record" classes that just hold data
from typing import List             # lets us write List[X] = "a list of X"

import autogen_core                 # the framework package (used here only to print its version)

# --- The core building blocks of any AutoGen application ---
from autogen_core import (
    AgentId,                        # an agent's ADDRESS: type (role) + key (which instance)
    CancellationToken,              # a "stop signal" you can pass into slow operations
    FunctionCall,                   # the model's REQUEST to run a tool: a name plus arguments
    MessageContext,                 # extra info for a handler: sender, topic, stop signal
    RoutedAgent,                    # the base class every agent in this notebook is built from
    SingleThreadedAgentRuntime,     # the ENVIRONMENT: creates agents, delivers messages
    TopicId,                        # the address of a "noticeboard" that agents can post to
    TypeSubscription,               # the rule that connects one noticeboard to one type of agent
    message_handler,                # the marker that turns a method into an inbox
)

# --- The message types that a language model understands ---
from autogen_core.models import (
    AssistantMessage,               # something the model said
    ChatCompletionClient,           # "any chat model" - used only as a type hint in our classes
    FunctionExecutionResult,        # the RESULT of one tool run, to be sent back to the model
    FunctionExecutionResultMessage, # a bundle of those results, grouped as models expect
    LLMMessage,                     # "any one of these message types" - another type hint
    ModelFamily,                    # names of model families AutoGen knows about
    ModelInfo,                      # a description of what a model can do (tools? vision? JSON?)
    SystemMessage,                  # the standing instruction that sets an agent's job = its GOAL
    UserMessage,                    # input from a person (or forwarded from another agent)
)

# --- Turning Python functions into things a model is allowed to call ---
# FunctionTool wraps a Python function so a model can call it; Tool is the type hint.
from autogen_core.tools import FunctionTool, Tool

# --- The client that actually sends HTTP requests to the model provider ---
from autogen_ext.models.openai import OpenAIChatCompletionClient

print("autogen-core version:", autogen_core.__version__)   # if this prints, the install worked

autogen-core version: 0.7.5


---
## Part A — Agents and the Environment (no LLM yet)

Before we bring in a model, let's see the two things AutoGen Core is really made of:
**an agent** (an object that reacts to a message) and **a runtime** (the environment that
delivers messages and owns the agents).

### Cell 4 — Define a message type

**What this cell does:** defines the *vocabulary*. In AutoGen Core, agents never call each
other's methods — they only exchange typed messages. A plain `@dataclass` is enough.

**Input:** none.

**Output:**
```
Greeting(name='Asha')
```

In [ ]:
# ===== Define a MESSAGE - the vocabulary agents use to talk =====

# "@dataclass" is a shortcut: Python writes the boring setup code for us, so this class
# becomes a simple container holding one piece of data called "name".
@dataclass
class Greeting:
    name: str        # one field, and it must be text ("str" = string = text)


# Creating one and printing it. Greeting(name="Asha") builds the message object.
# n = Greeting(name="Asha")
# print(n)
print(Greeting(name="Asha"))

Greeting(name='Asha')


### Cell 5 — Define your first agent

**What this cell does:** creates an agent class. Two things make it an agent:

1. it subclasses `RoutedAgent`
2. one method is decorated with `@message_handler`

The **type hint on the `message` argument is the routing rule** — this handler runs only for
`Greeting` messages. `ctx` (a `MessageContext`) carries metadata such as the sender and a
cancellation token. `self.id` is the agent's address: a `type` ("greeter") and a `key`
(which instance).

**Input:** none — this cell only defines the class.

**Output:**
```
GreeterAgent defined
```

In [ ]:
# ===== Define an AGENT - an object that reacts to a message =====

# "class GreeterAgent(RoutedAgent)" means: build a new kind of agent that inherits
# everything RoutedAgent already knows how to do (identity, mailbox, publishing).
class GreeterAgent(RoutedAgent):

    # __init__ runs once when this agent is created. It is the setup step.
    def __init__(self) -> None:
        super().__init__("Says hello")   # tell the parent class our description

    # @ message_handler marks the method below as an INBOX.
    @message_handler
    # The TYPE HINT does the routing: "message: Greeting" means this method runs
    # only for Greeting messages. "ctx" carries sender info and the stop signal.
    # "async def" means it can pause for slow work; "-> str" means it returns text.
    async def on_greeting(self, message: Greeting, ctx: MessageContext) -> str:
        # self.id is this agent's address. f"..." inserts values into the text.
        return f"Hello, {message.name}! I am agent type '{self.id.type}', key '{self.id.key}'."


print("GreeterAgent defined")   # the class now exists, but no agent has been created yet

GreeterAgent defined


### Cell 6 — Create the environment and send a message

**What this cell does:** the full life cycle of an AutoGen application in six lines.

- `SingleThreadedAgentRuntime()` — the environment
- `GreeterAgent.register(...)` — teach the runtime *how* to build a `greeter`. Note: it registers
  a **factory**, not an instance. Agents are created lazily, the first time a message arrives
  for them.
- `runtime.start()` — start the background message loop
- `runtime.send_message(...)` — a direct, one-to-one call that waits for a return value (RPC style)
- `stop_when_idle()` / `close()` — drain the queue and release resources

**Input:** a `Greeting(name="Class")` addressed to `AgentId("greeter", "default")`.

**Output:**
```
Hello, Class! I am agent type 'greeter', key 'default'.
```

lambda: GreeterAgent(): This is what's called a factory function. It's a small, anonymous function that, when called, produces a new instance of GreeterAgent. Crucially, the runtime does not create a GreeterAgent immediately when you call register. Instead, it stores this lambda function. The runtime will only call lambda: GreeterAgent() (effectively GreeterAgent()) to create an actual GreeterAgent instance later, the very first time a message is sent to an agent with the type='greeter' and a specific key (like 'default'). This is known as lazy instantiation, which helps optimize resource usage by only creating agents when they are actually needed

In summary, this line is telling the AutoGen runtime: "When you encounter a message for an agent of type 'greeter' that doesn't exist yet, here's the recipe (the lambda function) to create a new GreeterAgent instance for it."

In [ ]:
# ===== Build the ENVIRONMENT and send one message through it =====

# 1. Create the runtime - the world our agents will live in.
runtime = SingleThreadedAgentRuntime()

# 2. Teach the runtime how to build a "greeter".
#    Note we hand over a RECIPE, not an agent: "lambda: GreeterAgent()" is a tiny
#    function meaning "when you need one, call GreeterAgent()". The runtime creates
#    the actual agent later, the first time a message arrives for it.
#    "await" = wait for this step to finish before moving on.
await GreeterAgent.register(runtime, "greeter", lambda: GreeterAgent()) #factory function

# 3. Start the runtime's background message loop.
runtime.start()

# 4. Send one message directly to one agent and wait for its answer (like a phone call).
#    AgentId("greeter", "default") is the address: role "greeter", instance "default".
reply = await runtime.send_message(Greeting(name="Class"), AgentId("greeter", "default"))

# 5. Wait until no messages are left in flight, then shut the runtime down cleanly.
await runtime.stop_when_idle()
await runtime.close()

print(reply)   # whatever on_greeting returned

Hello, Class! I am agent type 'greeter', key 'default'.


---
## Part B — Goals: giving an agent a job

An agent's **goal** is not a special API object. It is the `SystemMessage` you hand to the model
plus the message types the agent chooses to handle. Keep goals narrow — one agent, one job.

Our first goal: *read a support ticket and classify it.*

### Cell 7 — Create the model client (pointed at Groq)

**What this cell does:** builds the object that actually talks to the model. Every agent that
needs to "think" gets one of these.

We keep using `OpenAIChatCompletionClient` and simply point it at Groq — three extra arguments
do the whole job:

- `base_url` — Groq's OpenAI-compatible endpoint instead of OpenAI's
- `api_key` — your Groq key
- `model_info` — for any non-OpenAI model, AutoGen cannot look up the model's capabilities, so
  you declare them. `function_calling: True` is the one that matters here: without it, AutoGen
  refuses to send tools in Cell 14.

**Provider vs. model.** Groq is the *provider* — it runs open-weights models very fast. It does
not host GPT-4o or Claude. So the model name is always an open model: `openai/gpt-oss-120b`,
`qwen/qwen3.6-27b`, `minimaxai/minimax-m2.7`. The `openai/` prefix here just means OpenAI
released those weights publicly; the call still goes to Groq.

**Model IDs change.** Groq retires models on a schedule, and a retired ID returns
`404 model_not_found`. Cell 8 lists exactly what your key can reach today — run it if this cell's
model is ever rejected, and pick a live ID from the list.

Swapping providers later (Together, Ollama, OpenRouter, OpenAI itself) is a change to these
three lines and nothing else — the agents never know.

**Input:** the `GROQ_API_KEY` from Cell 2.

**Output:**
```
Model client ready: openai/gpt-oss-120b @ https://api.groq.com/openai/v1
```

In [ ]:
# ===== Create the MODEL CLIENT - the object that talks to Groq =====

MODEL = "openai/gpt-oss-120b"                  # open-weights model Groq hosts; supports tools
GROQ_BASE_URL = "https://api.groq.com/openai/v1"   # Groq's address, in OpenAI-compatible format

model_client = OpenAIChatCompletionClient(
    model=MODEL,                               # which model to ask
    base_url=GROQ_BASE_URL,                    # WHERE to send it - this is what makes it Groq
    api_key=os.environ["GROQ_API_KEY"],        # your key, read back from the setting above
    temperature=0.2,                           # 0 = predictable, 1 = creative. Low suits triage.

    # For any non-OpenAI model, AutoGen cannot look up what the model can do,
    # so we declare it here. Each line answers one question about the model:
    model_info=ModelInfo(
        vision=False,                          # can it look at images?  -> no
        function_calling=True,                 # can it call tools?      -> yes (REQUIRED)
        json_output=True,                      # can it be asked for JSON? -> yes
        family=ModelFamily.UNKNOWN,            # a family AutoGen knows? -> no, that is fine
        structured_output=False,               # guarantee a fixed schema? -> no
    ),
)

# f"..." builds one line of text with the two values inserted.
print(f"Model client ready: {MODEL} @ {GROQ_BASE_URL}")

Model client ready: openai/gpt-oss-120b @ https://api.groq.com/openai/v1


### Cell 8 — Smoke test: what your key can reach, then one call

**What this cell does:** two checks in one.

1. It asks Groq which models your key can actually use. If `MODEL` from Cell 7 is missing from
   that list, copy a live id from the printout into Cell 7 and rerun — this is the fix for a
   `404 model_not_found`.
2. It then calls the model once, with no agents involved, to show the three message types you
   will keep seeing: `SystemMessage` (the instruction), `UserMessage` (the input), and the
   returned `CreateResult`.

A wrong key fails here with a 401 rather than deep inside an agent — that is the point of a
smoke test.

**Input:** the client from Cell 7.

**Output (model list and wording will vary):**
```
Chat models available to your key:
  groq/compound
  llama-3.1-8b-instant
  openai/gpt-oss-120b
  openai/gpt-oss-20b
  qwen/qwen3.6-27b
Is "openai/gpt-oss-120b" available? True

Reply: An agent is a program that pursues a goal by deciding its own next step.
Tokens used: RequestUsage(prompt_tokens=41, completion_tokens=18)
```

In [ ]:
# ===== SMOKE TEST: check the key, list the models, then make one real call =====

import httpx    # a library for making web requests - here we ask Groq "what models can I use?"

# PART 1 - ask Groq for the list of models this key can reach.
#   .get(...)  = fetch a web address
#   headers=   = how you prove who you are; "Bearer <key>" is the standard format
#   .json()    = turn the reply into a Python dictionary; ["data"] picks out the model list
models = httpx.get(
    f"{GROQ_BASE_URL}/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
    timeout=30,                                  # give up after 30 seconds instead of hanging
).json()["data"]

# Keep only the chat models (drop the speech-to-text ones) and sort them alphabetically.
ids = sorted(m["id"] for m in models if "whisper" not in m["id"])

print("Chat models available to your key:")
for model_id in ids:                             # a loop: do this once per model in the list
    print(" ", model_id)

# If this prints False, copy one of the ids above into MODEL in Cell 7 and run it again.
print(f'Is "{MODEL}" available?', MODEL in ids)
print()

# PART 2 - one real call to the model, with no agents involved yet.
# We hand it a list of two messages: the instruction, then the question.
result = await model_client.create(
    [
        SystemMessage(content="You explain ideas to beginners in one short sentence."),
        UserMessage(content="What is a software agent?", source="student"),
    ]
)

print("Reply:", result.content)      # .content is what the model actually said
print("Tokens used:", result.usage)  # .usage is how much text went in and came out

Chat models available to your key:
  allam-2-7b
  canopylabs/orpheus-arabic-saudi
  canopylabs/orpheus-v1-english
  groq/compound
  groq/compound-mini
  meta-llama/llama-prompt-guard-2-22m
  meta-llama/llama-prompt-guard-2-86m
  openai/gpt-oss-120b
  openai/gpt-oss-20b
  openai/gpt-oss-safeguard-20b
  qwen/qwen3.6-27b
  qwen/qwen3.8-27b
Is "openai/gpt-oss-120b" available? True

Reply: A software agent is a program that can act on its own to perform tasks for you.
Tokens used: RequestUsage(prompt_tokens=92, completion_tokens=95)


### Cell 9 — Define the application's messages

**What this cell does:** defines the three messages that flow through our support desk:

- `Ticket` — what the customer sent in
- `TriagedTicket` — the ticket after classification
- `DraftReply` — the answer we are ready to send back

> **Gotcha worth remembering:** AutoGen serialises dataclass messages, and a dataclass that
> *contains another dataclass* is not supported. Keep message classes flat (or use a Pydantic
> model). That is why `TriagedTicket` repeats the ticket fields instead of nesting a `Ticket`.

**Input:** none.

**Output:**
```
Ticket(ticket_id='T-501', customer='asha', text='Where is my order A1001?')
```

In [ ]:
# ===== The three MESSAGES that flow through our support desk =====
# Each one is a plain data container. Together they define the whole conversation.

@dataclass
class Ticket:                # what the customer originally sent in
    ticket_id: str           # e.g. "T-501" - also used as the conversation id
    customer: str            # who wrote it
    text: str                # what they wrote


@dataclass
class TriagedTicket:         # the same ticket AFTER the triage agent has classified it
    ticket_id: str
    customer: str
    text: str
    category: str            # added by triage: order_status / refund / technical / other
    urgency: str             # added by triage: low / medium / high


@dataclass
class DraftReply:            # the answer we are ready to send back to the customer
    ticket_id: str
    text: str


# Build one Ticket and print it, just to see the shape.
print(Ticket(ticket_id="T-501", customer="asha", text="Where is my order A1001?"))

Ticket(ticket_id='T-501', customer='asha', text='Where is my order A1001?')


### Cell 10 — The Triage agent (a goal in code)

**What this cell does:** defines an agent whose whole job is classification. Read it in three parts:

1. `__init__` stores the model client and the **goal** as a `SystemMessage`
2. the handler calls the model with that goal plus the customer's text
3. it then **publishes** a `TriagedTicket` to the topic `triaged_tickets` — it does not know or
   care which agent picks it up next. That is the loose coupling AutoGen Core is built around.

`ctx.cancellation_token` is passed along so the call can be cancelled if the runtime shuts down.

**Why the `extract_json` helper?** Open models such as Llama often wrap JSON in a
```` ```json ```` fence even when told not to. Rather than fight the prompt, we strip the fence
before parsing. Defensive parsing of model output is normal agent engineering, not a hack.

**Input:** a `Ticket` message (delivered later, in Part D).

**Output:** nothing yet — this cell only defines a helper and the class:
```
TriageAgent defined
```

 @message_handler is like putting a special label on a method in your agent. This label tells the AutoGen system: "Hey, if a message of this specific type arrives for this agent, this is the method that should handle it." It's how the agent knows which internal function to run when it receives a particular kind of message.

In [ ]:
# ===== AGENT 1: the triage agent - its GOAL is to classify a ticket =====

# A small helper. Models often wrap JSON in a ```json fence even when told not to,
# so we clean the text up before trying to read it.
def extract_json(text: str) -> dict:
    "Parse a JSON object out of a model reply, tolerating code fences and stray prose."
    text = text.strip()                       # remove blank space at the ends
    if text.startswith("```"):                # if the reply is fenced...
        text = text.split("```")[1]           # ...keep only what is between the fences
        text = text[4:] if text.startswith("json") else text   # drop the word "json" if present
    start, end = text.find("{"), text.rfind("}")   # find the first { and the last }
    return json.loads(text[start : end + 1])       # turn that slice into a Python dictionary


class TriageAgent(RoutedAgent):

    # Setup: remember the model client, and write down this agent's goal.
    def __init__(self, model_client: ChatCompletionClient) -> None:
        super().__init__("Classifies incoming support tickets")
        self._model_client = model_client     # "self._x" = something this agent keeps for later

        # THIS IS THE GOAL. A SystemMessage is a standing instruction the model
        # sees on every call. Notice how specific the output format request is.
        self._goal: List[LLMMessage] = [
            SystemMessage(
                content=(
                    "You are a support triage assistant. Read the customer's ticket and reply "
                    "with ONLY a JSON object, no markdown fences, in this exact shape:\n"
                    '{"category": "order_status|refund|technical|other", '
                    '"urgency": "low|medium|high"}'
                )
            )
        ]

    @message_handler                          # inbox for Ticket messages
    async def on_ticket(self, message: Ticket, ctx: MessageContext) -> None:

        # STEP 1: ask the model. We send the goal PLUS the customer's words.
        # The "+" joins two lists of messages into one.
        result = await self._model_client.create(
            self._goal + [UserMessage(content=message.text, source=message.customer)],
            cancellation_token=ctx.cancellation_token,   # pass the stop signal along
        )

        # STEP 2: turn the model's text answer into a usable dictionary.
        data = extract_json(str(result.content))
        print(f"[triage]  {message.ticket_id} -> {data}")

        # STEP 3: post the enriched ticket to the "triaged_tickets" noticeboard.
        # This agent does NOT know who reads it next - that is the whole point.
        await self.publish_message(
            TriagedTicket(
                ticket_id=message.ticket_id,
                customer=message.customer,
                text=message.text,
                category=data["category"],       # pull the two new fields out of the dictionary
                urgency=data["urgency"],
            ),
            topic_id=TopicId(type="triaged_tickets", source=message.ticket_id),
        )


print("TriageAgent defined")

TriageAgent defined


---
## Part C — Actions: letting an agent do things

A model on its own can only produce text. An **action** is a Python function you expose to the
model as a *tool*. The model chooses the tool and the arguments; **your code** actually runs it.
That boundary is what keeps agents safe and testable.

### Cell 11 — Write plain Python functions

**What this cell does:** defines our "backend". `ORDERS` stands in for a real database, and the
two functions are ordinary Python — nothing AutoGen-specific yet. Notice they return JSON strings,
which models read reliably.

**Input:** an order id / a product category.

**Output:**
```
{"order_id": "A1001", "status": "shipped", "carrier": "BlueDart", "eta": "2026-09-03", "total": 2499.0}
No order found with id A9999.
```

In [ ]:
# ===== The ACTIONS, part 1: ordinary Python functions =====
# Nothing here is AutoGen-specific yet. This is just our pretend backend.

# A dictionary standing in for a real orders database.
# Read it as: order "A1001" -> these details.
ORDERS = {
    "A1001": {"status": "shipped", "carrier": "BlueDart", "eta": "2026-09-03", "total": 2499.0},
    "A1002": {"status": "processing", "carrier": None, "eta": "2026-09-06", "total": 899.0},
}


def get_order_status(order_id: str) -> str:
    "Look up one order in the store's order book."
    order = ORDERS.get(order_id.upper())          # .upper() so "a1001" also works
    if order is None:                             # .get() returns None when nothing matches
        return f"No order found with id {order_id}."
    # json.dumps turns a dictionary into JSON text. Models read JSON far more reliably
    # than sentences, so tools should return JSON whenever they can.
    # The "**order" part means "and all the fields from the order as well".
    return json.dumps({"order_id": order_id.upper(), **order})


def get_refund_policy(category: str) -> str:
    "Return the refund rules that apply to a product category."
    return json.dumps(
        {"category": category, "window_days": 30, "condition": "unused, in original packaging"}
    )


# Try both functions by hand - one hit, one miss.
print(get_order_status("A1001"))
print(get_order_status("A9999"))

{"order_id": "A1001", "status": "shipped", "carrier": "BlueDart", "eta": "2026-09-03", "total": 2499.0}
No order found with id A9999.


### Cell 12 — Turn the functions into tools

**What this cell does:** wraps each function in a `FunctionTool`. AutoGen reads the signature and
type hints and generates a JSON schema — that schema is what the model sees. This is why clear
parameter names and a good `description` matter: they are the instructions the model reads.

**Input:** the two functions from Cell 11.

**Output:**
```
{
  "name": "get_order_status",
  "description": "Look up the status of a customer order by order id.",
  "parameters": {
    "type": "object",
    "properties": {
      "order_id": {
        "description": "order_id",
        "title": "Order Id",
        "type": "string"
      }
    },
    "required": ["order_id"],
    "additionalProperties": false
  },
  "strict": false
}
```

In [ ]:
# ===== The ACTIONS, part 2: wrap the functions as TOOLS =====
# FunctionTool reads the function's name, its arguments and their types, and builds
# the description the model will see. The "description" you write is an instruction
# to the model about when to use this tool - so it is worth writing carefully.

order_tool = FunctionTool(
    get_order_status, description="Look up the status of a customer order by order id."
)
policy_tool = FunctionTool(
    get_refund_policy, description="Return the refund policy for a product category."
)

# Put both tools in one list, which we will hand to the support agent later.
support_tools: List[Tool] = [order_tool, policy_tool]

# Print the generated description. THIS is what actually travels to the model -
# your function has become a form the model knows how to fill in.
# (indent=2 just makes the JSON readable.)
print(json.dumps(order_tool.schema, indent=2))

{
  "name": "get_order_status",
  "description": "Look up the status of a customer order by order id.",
  "parameters": {
    "type": "object",
    "properties": {
      "order_id": {
        "description": "order_id",
        "title": "Order Id",
        "type": "string"
      }
    },
    "required": [
      "order_id"
    ],
    "additionalProperties": false
  },
  "strict": false
}


### Cell 13 — Run a tool by hand

**What this cell does:** calls the tool the same way the framework will, with a JSON dict of
arguments. Useful for debugging: if a tool misbehaves here, the problem is your function, not
the model. The `CancellationToken` lets a long-running tool be interrupted.

**Input:** `{"order_id": "a1002"}`.

**Output:**
```
{"order_id": "A1002", "status": "processing", "carrier": null, "eta": "2026-09-06", "total": 899.0}
```

In [ ]:
# ===== Run a tool by hand, exactly the way the framework will =====
# Useful for debugging: if a tool misbehaves HERE, the problem is your function,
# not the model.

raw = await order_tool.run_json(
    {"order_id": "a1002"},    # the arguments, as a dictionary - the same shape the model sends
    CancellationToken(),      # a fresh stop signal (nothing to cancel here)
    call_id="manual-1",       # a label for this call; the model normally generates one
)

# The tool returns a Python value; this turns it into the text the model would receive.
print(order_tool.return_value_as_string(raw))

{"order_id": "A1002", "status": "processing", "carrier": null, "eta": "2026-09-06", "total": 899.0}


### Cell 14 — The Support agent (goal + actions in one loop)

**What this cell does:** this is the heart of the notebook — the *tool-use loop*:

1. send the conversation **plus the tool schemas** to the model
2. if the model answers with text → we are done
3. if it answers with `FunctionCall` objects → run those tools, append the results as a
   `FunctionExecutionResultMessage`, and ask the model again
4. repeat until the model produces text

The `while` loop matters: a model may need several rounds (look up the order, *then* check the
refund policy). Finally the agent publishes a `DraftReply` to the `draft_replies` topic.

Models on Groq can also return **several tool calls in one reply**, which is why
`result.content` is a *list* of `FunctionCall` objects and we execute all of them before
replying.

**Input:** a `TriagedTicket` message (delivered in Part D).

**Output:**
```
SupportAgent defined
```

In [ ]:
# ===== AGENT 2: the support agent - GOAL plus ACTIONS, in a loop =====

class SupportAgent(RoutedAgent):

    # Setup: this agent needs both a model AND a list of tools it may use.
    def __init__(self, model_client: ChatCompletionClient, tools: List[Tool]) -> None:
        super().__init__("Answers tickets using tools")
        self._model_client = model_client
        self._tools = tools

    @message_handler                     # inbox for TriagedTicket messages
    async def on_triaged(self, message: TriagedTicket, ctx: MessageContext) -> None:

        # STEP 1: start a conversation - the goal, then the customer's ticket.
        # "session" is simply the running transcript; we add to it as we go.
        session: List[LLMMessage] = [
            SystemMessage(
                content=(
                    "You are a customer support agent for an online store. "
                    "Use the tools to get facts before answering - never invent order details. "
                    "Then write a friendly reply to the customer in 2-3 sentences."
                )
            ),
            UserMessage(
                content=f"[category={message.category}, urgency={message.urgency}] {message.text}",
                source=message.customer,
            ),
        ]

        # STEP 2: ask the model, and this time send the TOOLS along with the question.
        result = await self._model_client.create(
            session, tools=self._tools, cancellation_token=ctx.cancellation_token
        )

        # STEP 3: the tool loop.
        # If the model replied with TEXT, we are done and this loop never runs.
        # If it replied with a LIST, that list is tool requests - so we run them,
        # add the results to the transcript, and ask again. Repeat until we get text.
        while isinstance(result.content, list):

            # a) record what the model asked for
            session.append(AssistantMessage(content=result.content, source="assistant"))

            # b) actually run each requested tool. This is a compact way of writing a loop:
            #    "for every call in the list, run it, and collect the answers".
            tool_results = [
                await self._run_tool(call, ctx.cancellation_token) for call in result.content
            ]

            # c) hand the results back to the model, grouped the way it expects
            session.append(FunctionExecutionResultMessage(content=tool_results))

            # d) ask again, now that it has the facts
            result = await self._model_client.create(
                session, tools=self._tools, cancellation_token=ctx.cancellation_token
            )

        # STEP 4: the loop ended, so result.content is the final text. Post it.
        await self.publish_message(
            DraftReply(ticket_id=message.ticket_id, text=str(result.content)),
            topic_id=TopicId(type="draft_replies", source=message.ticket_id),
        )

    # A private helper (the leading "_" is a convention meaning "internal use").
    # It runs ONE tool request and packages the answer for the model.
    async def _run_tool(
        self, call: FunctionCall, token: CancellationToken
    ) -> FunctionExecutionResult:

        # Find the tool whose name matches what the model asked for.
        tool = next(t for t in self._tools if t.name == call.name)
        print(f"[action]  {call.name}({call.arguments})")

        # "try / except" = attempt this, and if it fails, do that instead - so a broken
        # tool becomes a message the model can react to rather than a crash.
        try:
            # call.arguments arrives as JSON TEXT, so json.loads turns it into a dictionary.
            output = await tool.run_json(json.loads(call.arguments), token, call_id=call.id)
            return FunctionExecutionResult(
                call_id=call.id,                             # ties the answer to the request
                content=tool.return_value_as_string(output), # the result, as text
                is_error=False,
                name=tool.name,
            )
        except Exception as exc:      # "exc" holds the error that happened
            return FunctionExecutionResult(
                call_id=call.id, content=str(exc), is_error=True, name=tool.name
            )


print("SupportAgent defined")

SupportAgent defined


---
## Part D — Environment: topics, subscriptions and the runtime

`send_message` is a phone call: you need the callee's address and you wait for an answer.
`publish_message` is a noticeboard: you pin a message to a **topic**, and every agent that
**subscribed** to that topic gets a copy. Our pipeline uses the noticeboard, so we can add a
logging agent later without touching a single existing agent.

A `TopicId` has a **type** (`"new_tickets"`) and a **source** (we use the ticket id).
With a `TypeSubscription`, the runtime creates **one agent instance per source** — so ticket
T-501 and ticket T-502 are handled by separate agent instances that never mix state.

### Cell 15 — The Supervisor agent (end of the line)

**What this cell does:** collects finished replies. Because each ticket gets its own agent
instance, we append to a module-level `OUTBOX` list so we can inspect everything afterwards.
In a real system this is where you would email the customer or push to a review queue.

**Input:** `DraftReply` messages.

**Output:**
```
SupervisorAgent defined
```

In [ ]:
# ===== AGENT 3: the supervisor - the end of the line =====

# A plain list that lives outside the agent. We need it because the runtime creates
# ONE supervisor per ticket, so anything stored inside an agent would be split up.
OUTBOX: List[DraftReply] = []


class SupervisorAgent(RoutedAgent):

    def __init__(self) -> None:
        super().__init__("Collects finished replies for human review")

    @message_handler                  # inbox for DraftReply messages
    async def on_reply(self, message: DraftReply, ctx: MessageContext) -> None:
        OUTBOX.append(message)        # .append() adds one item to the end of the list
        print(f"[outbox]  {message.ticket_id} ready for review")
        # In a real system this is where you would email the customer
        # or push the draft into a review queue.


print("SupervisorAgent defined")

SupervisorAgent defined


### Cell 16 — Wire up the application

**What this cell does:** builds the environment for the real application:

- register the three agent types (three factories)
- add three subscriptions, which define the pipeline:

```
new_tickets ──► triage_agent ──► triaged_tickets ──► support_agent ──► draft_replies ──► supervisor
```

Read the subscriptions as sentences: *"topic `new_tickets` is delivered to agents of type
`triage_agent`."*

**Input:** the classes defined above and `model_client` from Cell 7.

**Output:**
```
Registered: ('triage_agent', 'support_agent', 'supervisor')
Subscriptions: new_tickets -> triage_agent -> triaged_tickets -> support_agent -> draft_replies -> supervisor
```

Agent per Source (AutoGen Core): In AutoGen Core, when an agent is created "per source" (e.g., per ticket_id), it means:

    Dedicated Instance: Each unique ticket gets its own, independent agent instance. This agent can maintain state (e.g., remember the conversation history for that specific ticket) without interfering with other tickets.

    Concurrency: Multiple agents for different sources can run at the same time within the runtime, allowing many tickets to be processed in parallel or near-parallel.

    Fault Isolation: If one agent instance encounters an issue while processing T-501, it doesn't affect the agent handling T-502.


In [ ]:
# ===== Wire the APPLICATION together =====

# 1. A fresh environment for the real application.
desk = SingleThreadedAgentRuntime()

# 2. Register the recipe for each of the three agents.
#    Nothing is created yet - these are instructions for later.
await TriageAgent.register(desk, "triage_agent", lambda: TriageAgent(model_client))
await SupportAgent.register(
    desk, "support_agent", lambda: SupportAgent(model_client, support_tools)
)
await SupervisorAgent.register(desk, "supervisor", lambda: SupervisorAgent())

# 3. Connect the noticeboards to the agents. Read each line as a sentence:
#    "anything posted to topic X is delivered to agents of type Y".
#    These three lines ARE the pipeline - the only place its shape is defined.
await desk.add_subscription(TypeSubscription(topic_type="new_tickets", agent_type="triage_agent"))
await desk.add_subscription(
    TypeSubscription(topic_type="triaged_tickets", agent_type="support_agent")
)
await desk.add_subscription(TypeSubscription(topic_type="draft_replies", agent_type="supervisor"))

print("Registered:", ("triage_agent", "support_agent", "supervisor"))
print(
    "Subscriptions: new_tickets -> triage_agent -> triaged_tickets -> "
    "support_agent -> draft_replies -> supervisor"
)

Registered: ('triage_agent', 'support_agent', 'supervisor')
Subscriptions: new_tickets -> triage_agent -> triaged_tickets -> support_agent -> draft_replies -> supervisor


### Cell 17 — Run the application

**What this cell does:** starts the runtime and drops two tickets onto the `new_tickets` topic,
then waits for the whole cascade to finish. Nobody calls anybody directly — the runtime routes
every hop. The two tickets are processed concurrently.

**Input:** two `Ticket` messages (one order question, one refund question).

**Output (exact wording from the model will vary):**
```
[triage]  T-501 -> {'category': 'order_status', 'urgency': 'medium'}
[triage]  T-502 -> {'category': 'refund', 'urgency': 'low'}
[action]  get_order_status({"order_id":"A1001"})
[action]  get_refund_policy({"category":"headphones"})
[outbox]  T-501 ready for review
[outbox]  T-502 ready for review
Done. 2 replies drafted.
```

In [ ]:
# ===== Run the application =====

# Two pretend customer emails - one about delivery, one about a return.
tickets = [
    Ticket(
        ticket_id="T-501",
        customer="asha",
        text="Hi, where is my order A1001? It was supposed to arrive yesterday.",
    ),
    Ticket(
        ticket_id="T-502",
        customer="ravi",
        text="The headphones I bought are fine but I changed my mind. Can I still return them?",
    ),
]

desk.start()                       # switch the message loop on

# Post each ticket to the "new_tickets" noticeboard and walk away.
# We never name an agent here - the subscriptions decide who wakes up.
# "source=ticket.ticket_id" keeps each ticket in its own private lane.
for ticket in tickets:
    await desk.publish_message(
        ticket, topic_id=TopicId(type="new_tickets", source=ticket.ticket_id)
    )

# Wait for the whole chain to finish: triage -> support -> tools -> supervisor.
# Both tickets are handled at the same time, so the printed lines may interleave.
await desk.stop_when_idle()

print(f"Done. {len(OUTBOX)} replies drafted.")   # len() = how many items in the list

[triage]  T-502 -> {'category': 'refund', 'urgency': 'low'}
[triage]  T-501 -> {'category': 'order_status', 'urgency': 'high'}
[action]  get_order_status({"order_id":"A1001"})
[action]  get_refund_policy({"category":"headphones"})
[outbox]  T-502 ready for review
[outbox]  T-501 ready for review
Done. 2 replies drafted.


### Cell 18 — Read the finished work

**What this cell does:** prints what the pipeline produced. This is the "application" boundary —
the point where agent output becomes something a human or another system uses.

**Input:** the `OUTBOX` list filled in during Cell 17.

**Output (wording will vary):**
```
--- T-501 ---
Thanks for checking in! Your order A1001 has shipped with BlueDart and is expected on
2026-09-03. Sorry for the short delay — the tracking should update shortly.

--- T-502 ---
Of course! Headphones can be returned within 30 days as long as they are unused and in the
original packaging. Just reply here and I'll start the return for you.
```

In [ ]:
# ===== Read the finished work =====
# This is the "application boundary": where agent output becomes something a human uses.

for reply in OUTBOX:                             # go through the collected replies one by one
    print(f"--- {reply.ticket_id} ---")
    print(reply.text)
    print()                                      # blank line between replies

--- T-502 ---
Sure thing! You can return the headphones within 30 days as long as they’re still unused and in their original packaging. Just let us know when you’d like to start the return, and we’ll guide you through the next steps.

--- T-501 ---
Your order A1001 has already been shipped via BlueDart and is expected to arrive on September 3. We’re sorry it didn’t reach you yesterday, and we’ll keep an eye on its progress to ensure a smooth delivery. If you need any further assistance, just let us know!



### Cell 19 — Shut down cleanly

**What this cell does:** releases the runtime and closes the HTTP connection behind the model
client. Always do this at the end of a script or notebook — it is the equivalent of closing a
database connection.

**Input:** the live `desk` runtime and `model_client`.

**Output:**
```
Runtime closed. Model client closed.
```

In [ ]:
# ===== Shut everything down cleanly =====
# Same idea as closing a file or a database connection when you are finished.

await desk.close()           # release the runtime and its agents
await model_client.close()   # close the network connection to Groq

print("Runtime closed. Model client closed.")

---
## Recap — what you just built

```
  Ticket                TriagedTicket              DraftReply
    │                        │                         │
    ▼                        ▼                         ▼
TriageAgent  ──publish──► SupportAgent ──publish──► SupervisorAgent
  (goal:                  (goal + actions:            (goal:
   classify)               tools + LLM loop)           collect)
    └──────────── SingleThreadedAgentRuntime ─────────────┘
                    (the environment)
```

| Concept | The line of code that shows it |
|---|---|
| Goal | `SystemMessage(content="You are a support triage assistant...")` |
| Action | `FunctionTool(get_order_status, description=...)` |
| Agent | `class TriageAgent(RoutedAgent)` + `@message_handler` |
| Environment | `SingleThreadedAgentRuntime()`, `TypeSubscription(...)`, `TopicId(...)` |
| Application | Cells 16-18 — register, subscribe, publish, collect |

## Try it yourself

1. **A new action.** Add a `create_return_label(order_id)` tool and let the support agent use it
   for refund tickets.
2. **A new agent, zero edits elsewhere.** Write an `AuditAgent` that also subscribes to
   `draft_replies` and logs every reply to a file. You should not need to change any existing
   agent — that is the payoff of the topic-based design.
3. **A ticket that has no answer.** Send `"Where is my order A9999?"` and watch how the agent
   handles a tool result that says the order does not exist.
4. **Escalation.** In `TriageAgent`, publish `urgency == "high"` tickets to a different topic and
   give that topic its own agent.
5. **Swap the model.** Change `MODEL` in Cell 7 to `openai/gpt-oss-20b` (or another id from the
   Cell 8 list) and rerun the pipeline. Does the smaller model still pick the right tool and the
   right order id?

## Where to go next

- **AgentChat** (`autogen-agentchat`) — a higher-level layer built on this core, with ready-made
  `AssistantAgent` and team patterns like `RoundRobinGroupChat`. Good for prototyping once you
  understand what is underneath.
- **Distributed runtime** — the same agent code runs on a gRPC-backed runtime across processes
  and machines; only the runtime object changes.
- **Another model or provider** — only Cell 7 changes. Try `openai/gpt-oss-20b` or a Qwen model
  on Groq, or point `base_url` at any other OpenAI-compatible endpoint. A weaker model usually
  fails first at tool choice — a good thing to observe deliberately.
- **Docs** — https://microsoft.github.io/autogen/stable/user-guide/core-user-guide/